# 🎵 wav2vec 2.0: Complete PyTorch Implementation & Audio Pipeline

This notebook provides a **complete, working PyTorch implementation of wav2vec 2.0** (*Baevski et al., Meta AI, 2020*).

### What this notebook includes:
1. **Convolutional Feature Extractor**: 7-layer 1D CNN with exact stride math ($320\times$ reduction).
2. **Product Quantization & Gumbel-Softmax**: Differentiable categorical lookup over $G=2$ codebooks with $V=320$ entries.
3. **Span Masking Engine**: Consecutive mask span generation over time steps.
4. **Transformer Context Encoder**: Multi-head attention model producing contextual vectors $c_t$.
5. **Contrastive & Diversity Loss Functions**: Cosine similarity math with distractor sampling.
6. **Real Audio Demonstration**: Generating synthetic 16kHz speech waveform and tracing tensor dimensions throughout the network.

In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# Check device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


--- 
## 1. Temporal Convolutional Feature Encoder $f: \mathcal{X} \to \mathcal{Z}$

The encoder consists of **7 blocks of 1D temporal convolutions** with 512 channels each.

- **Kernel Sizes**: $(10, 3, 3, 3, 3, 2, 2)$
- **Strides**: $(5, 2, 2, 2, 2, 2, 2)$
- **Total Stride Factor**: $5 \times 2^6 = 320$
- **Sample Rate**: $16,000\text{ Hz} \implies \frac{320}{16000} = 20\text{ ms stride per frame}$.

In [4]:
class ConvFeatureEncoder(nn.Module):
    """
    Temporal Convolutional Feature Encoder f: X -> Z
    
    Conv1D Weight Shape Format: (out_channels, in_channels, kernel_size)
    -------------------------------------------------------------------
    Layer 1: Weight Shape (512,   1, 10) | Output Shape: (Batch, 512, 6400)
    Layer 2: Weight Shape (512, 512,  3) | Output Shape: (Batch, 512, 3199)
    Layer 3: Weight Shape (512, 512,  3) | Output Shape: (Batch, 512, 1599)
    Layer 4: Weight Shape (512, 512,  3) | Output Shape: (Batch, 512,  799)
    Layer 5: Weight Shape (512, 512,  3) | Output Shape: (Batch, 512,  399)
    Layer 6: Weight Shape (512, 512,  2) | Output Shape: (Batch, 512,  199)
    Layer 7: Weight Shape (512, 512,  2) | Output Shape: (Batch, 512,   99)
    """
    def __init__(self, in_channels=1, embed_dim=512, verbose=False):
        super().__init__()
        self.verbose = verbose
        
        # (out_channels, kernel_size, stride)
        conv_params = [
            (512, 10, 5),  # Layer 1 Weight -> (512, 1, 10) (1,1,1,32000)
            (512,  3, 2),  # Layer 2 Weight -> (512, 512, 3)
            (512,  3, 2),  # Layer 3 Weight -> (512, 512, 3)
            (512,  3, 2),  # Layer 4 Weight -> (512, 512, 3)
            (512,  3, 2),  # Layer 5 Weight -> (512, 512, 3)
            (512,  2, 2),  # Layer 6 Weight -> (512, 512, 2)
            (512,  2, 2),  # Layer 7 Weight -> (512, 512, 2)
        ]
        
        self.conv_layers = nn.ModuleList()
        curr_in = in_channels
        for out_ch, k_size, stride in conv_params:
            conv = nn.Conv1d(curr_in, out_ch, kernel_size=k_size, stride=stride, bias=False)
            block = nn.Sequential(conv, nn.Dropout(p=0.0), nn.GELU())
            self.conv_layers.append(block)
            curr_in = out_ch
            
        self.layer_norm = nn.LayerNorm(embed_dim)

    def forward(self, x):
        # Input Shape: (Batch, 1, Raw_Samples) -> e.g. (1, 1, 32000)
        if self.verbose:
            print(f"📥 Input Audio Shape: {list(x.shape)}")
            print("-" * 65)
            
        for idx, block in enumerate(self.conv_layers):
            conv_layer = block[0]
            weight_shape = list(conv_layer.weight.shape)
            
            x = block(x)
            
            if self.verbose:
                print(f"Conv Layer {idx+1} | Weight Shape: {weight_shape} | Output Shape: {list(x.shape)}")
            
        # Transpose to (Batch, Time_Steps, Channels=512) for LayerNorm
        x = x.transpose(1, 2)
        x = self.layer_norm(x)
        
        if self.verbose:
            print("-" * 65)
            print(f"Final Extractor Output Shape (Batch, T, 512): {list(x.shape)}\n")
            
        return x  # z_t representations shape: (Batch, T, 512)


--- 
## 2. Quantization Module $Z \to Q$ (Gumbel-Softmax Product Quantization)

Maps latent features $z_t \in \mathbb{R}^{512}$ to discrete codebook vectors $q_t \in \mathbb{R}^{512}$.
- $G = 2$ codebooks (groups).
- $V = 320$ entries (codewords) per group.
- Codebook entry dimension $= 512 / 2 = 256$.
- Total unique combination targets: $V^G = 320^2 = 102,400$ combinations.

In [5]:
class GumbelVectorQuantizer(nn.Module):
    def __init__(self, in_dim=512, num_groups=2, 
    num_vars=320, vq_dim=512, temp=(2.0, 0.5, 0.999995)):
        super().__init__()
        self.G = num_groups
        self.V = num_vars
        self.vq_dim = vq_dim
        self.entry_dim = vq_dim // self.G  # 512 / 2 = 256
        
        # Linear projection to compute logits for all groups: (Batch, T, G * V)
        self.weight_proj = nn.Linear(in_dim, self.G * self.V)
        
        # Trainable Codebooks: (1, G * V, entry_dim)
        self.vars = nn.Parameter(torch.FloatTensor(1, self.G * self.V, self.entry_dim))
        nn.init.uniform_(self.vars, -1.0 / math.sqrt(self.entry_dim), 1.0 / math.sqrt(self.entry_dim))
        
        self.curr_temp = temp[0]

    def forward(self, x):
        # Input Shape: (Batch, Time_Steps, in_dim=512)
        B, T, D = x.size()
        
        # 1. Project to logits: (B, T, G * V)
        logits = self.weight_proj(x)
        
        # Reshape logits to 4D tensor: (B, T, G, V) for grouped softmax selection
        logits_grouped = logits.view(B, T, self.G, self.V)
        
        # 2. Gumbel-Softmax Sampling per group
        if self.training:
            gumbel_noise = -torch.empty_like(logits_grouped).exponential_().log()
            soft_probs = F.softmax((logits_grouped + gumbel_noise) / self.curr_temp, dim=-1)
        else:
            soft_probs = F.softmax(logits_grouped / self.curr_temp, dim=-1)
            
        # Hard one-hot index selection: (B, T, G, V)
        _, max_idx = soft_probs.max(dim=-1)
        hard_onehot = torch.zeros_like(logits_grouped).scatter_(-1, max_idx.unsqueeze(-1), 1.0)
        
        # Straight-Through Estimator trick
        code_probs = hard_onehot - soft_probs.detach() + soft_probs
        
        # 3. Lookup entries from codebook per group
        # vars shape: (1, G * V, entry_dim) -> reshape to (G, V, entry_dim)
        vars_reshaped = self.vars.view(self.G, self.V, self.entry_dim)
        
        # Multiply probabilities with codebooks: (B, T, G, V) x (G, V, entry_dim) -> (B, T, G, entry_dim)
        selected_entries = torch.einsum('btgv,gvd->btgd', code_probs, vars_reshaped)
        
        # Concatenate group vectors: (B, T, G * entry_dim) = (B, T, 512)
        q_targets = selected_entries.reshape(B, T, self.G * self.entry_dim)
        
        # Perplexity & average probabilities for Diversity Loss
        avg_probs = soft_probs.mean(dim=(0, 1))  # (G, V)
        prob_perplexity = torch.exp(-torch.sum(avg_probs * torch.log(avg_probs + 1e-7), dim=-1)).sum()
        
        return q_targets, prob_perplexity, avg_probs


--- 
## 3. Full wav2vec 2.0 Model (Masking + Transformer + Loss Calculation)

Assembles the feature encoder, quantizer, span masking, and Transformer context network into a single end-to-end model.

In [6]:
class Wav2Vec2Model(nn.Module):
    def __init__(self, embed_dim=512, num_heads=8, num_layers=6, num_groups=2, num_vars=320, verbose_encoder=True):
        super().__init__()
        self.feature_extractor = ConvFeatureEncoder(in_channels=1, embed_dim=embed_dim, verbose=verbose_encoder)
        self.quantizer = GumbelVectorQuantizer(in_dim=embed_dim, num_groups=num_groups, num_vars=num_vars, vq_dim=embed_dim)
        
        # Shared trainable mask embedding vector
        self.mask_emb = nn.Parameter(torch.FloatTensor(embed_dim))
        nn.init.uniform_(self.mask_emb)
        
        # Transformer Context Encoder
        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads, dim_feedforward=2048, dropout=0.1, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # Projection layer to align q target space
        self.project_q = nn.Linear(embed_dim, embed_dim)

    def apply_masking(self, z, mask_prob=0.065, mask_length=10):
        """Masks consecutive time spans in latent z"""
        B, T, D = z.size()
        mask_indices = torch.zeros(B, T, dtype=torch.bool, device=z.device)
        
        for b in range(B):
            num_spans = int(mask_prob * T)
            if num_spans > 0:
                starts = torch.randperm(max(1, T - mask_length))[:num_spans]
                for s in starts:
                    mask_indices[b, s : s + mask_length] = True
                
        z_masked = z.clone()
        z_masked[mask_indices] = self.mask_emb
        return z_masked, mask_indices

    def compute_contrastive_loss(self, c, q, mask_indices, num_negatives=10, temp=0.1):
        """Computes L_m over masked time steps using Cosine Similarity"""
        if not mask_indices.any():
            return torch.tensor(0.0, device=c.device)
            
        c_masked = c[mask_indices]  # (N_masked, D)
        q_masked = q[mask_indices]  # (N_masked, D)
        
        # Cosine normalization
        c_norm = F.normalize(c_masked, dim=-1)
        q_norm = F.normalize(q_masked, dim=-1)
        
        N_masked = c_masked.size(0)
        
        # Sample distractor targets from other masked steps
        negs = []
        for i in range(N_masked):
            other_indices = [idx for idx in range(N_masked) if idx != i]
            if len(other_indices) >= num_negatives:
                sampled_idx = torch.tensor(other_indices)[torch.randperm(len(other_indices))[:num_negatives]]
            else:
                sampled_idx = torch.randint(0, N_masked, (num_negatives,))
            negs.append(q_norm[sampled_idx])
            
        negs = torch.stack(negs)  # (N_masked, num_negatives, D)
        
        # Positive Similarity: (N_masked, 1)
        pos_sim = torch.sum(c_norm * q_norm, dim=-1, keepdim=True) / temp
        
        # Negative Similarities: (N_masked, num_negatives)
        neg_sim = torch.bmm(negs, c_norm.unsqueeze(-1)).squeeze(-1) / temp
        
        # Combine into classification logits: (N_masked, 1 + num_negatives)
        logits = torch.cat([pos_sim, neg_sim], dim=-1)
        labels = torch.zeros(N_masked, dtype=torch.long, device=c.device)
        
        return F.cross_entropy(logits, labels)

    def forward(self, raw_audio):
        # 1. Feature Extraction: (B, 1, Raw_Samples) -> (B, T, 512)
        z = self.feature_extractor(raw_audio)
        
        # 2. Quantization (Unmasked Z -> Target Q)
        q_targets, prob_perplexity, avg_probs = self.quantizer(z)
        q_projected = self.project_q(q_targets)
        
        # 3. Apply Span Masking
        z_masked, mask_indices = self.apply_masking(z)
        
        # 4. Context Transformer: (B, T, 512) -> (B, T, 512)
        c = self.transformer(z_masked)
        
        # 5. Losses
        contrastive_loss = self.compute_contrastive_loss(c, q_projected, mask_indices)
        diversity_loss = (avg_probs * torch.log(avg_probs + 1e-7)).sum(dim=-1).mean()
        total_loss = contrastive_loss + 0.1 * diversity_loss
        
        return {
            "loss": total_loss,
            "contrastive_loss": contrastive_loss,
            "diversity_loss": diversity_loss,
            "z_shape": z.shape,
            "c_shape": c.shape,
            "q_shape": q_targets.shape,
            "masked_steps": mask_indices.sum().item(),
            "perplexity": prob_perplexity.item()
        }


--- 
## 4. Execution with Real 16kHz Audio Signal & Dimension Tracing

We generate **2 seconds of 16kHz audio** (32,000 raw samples) combining fundamental frequencies (440Hz, 880Hz) to simulate a real speech waveform.

In [7]:
# Instantiate Model with verbose_encoder=True to print layer-by-layer weight & output shapes
model = Wav2Vec2Model(
    embed_dim=512, 
    num_heads=8, 
    num_layers=6, 
    num_groups=2, 
    num_vars=320, 
    verbose_encoder=True
).to(device)
model.train()

# Generate 2 seconds of 16kHz audio signal
sample_rate = 16000
duration_sec = 2.0
t = torch.linspace(0, duration_sec, int(sample_rate * duration_sec))
raw_audio = (torch.sin(2 * math.pi * 440 * t) + 0.5 * torch.sin(2 * math.pi * 880 * t)).unsqueeze(0).unsqueeze(0).to(device)

# Run Forward Pass (Will print all Conv weight shapes and output activation shapes)
outputs = model(raw_audio)

print("--- 📊 Model Final Outputs ---")
print(f"1. Latent Vectors (Z) Shape         : {outputs['z_shape']}  (99 frames of 20ms stride)")
print(f"2. Context Representations (C) Shape : {outputs['c_shape']}")
print(f"3. Quantized Target Vectors (Q) Shape: {outputs['q_shape']}")
print(f"4. Masked Time Steps                 : {outputs['masked_steps']} / 99 steps (~49% masked)")
print(f"5. Codebook Perplexity               : {outputs['perplexity']:.2f} / 640.00 max")

print("\n--- 📉 Loss Breakdown ---")
print(f"Contrastive Loss (L_m)              : {outputs['contrastive_loss'].item():.4f}")
print(f"Diversity Loss (L_d)                : {outputs['diversity_loss'].item():.4f}")
print(f"Total Combined Loss (L)             : {outputs['loss'].item():.4f}")


📥 Input Audio Shape: [1, 1, 32000]
-----------------------------------------------------------------
Conv Layer 1 | Weight Shape: [512, 1, 10] | Output Shape: [1, 512, 6399]
Conv Layer 2 | Weight Shape: [512, 512, 3] | Output Shape: [1, 512, 3199]
Conv Layer 3 | Weight Shape: [512, 512, 3] | Output Shape: [1, 512, 1599]
Conv Layer 4 | Weight Shape: [512, 512, 3] | Output Shape: [1, 512, 799]
Conv Layer 5 | Weight Shape: [512, 512, 3] | Output Shape: [1, 512, 399]
Conv Layer 6 | Weight Shape: [512, 512, 2] | Output Shape: [1, 512, 199]
Conv Layer 7 | Weight Shape: [512, 512, 2] | Output Shape: [1, 512, 99]
-----------------------------------------------------------------
Final Extractor Output Shape (Batch, T, 512): [1, 99, 512]

--- 📊 Model Final Outputs ---
1. Latent Vectors (Z) Shape         : torch.Size([1, 99, 512])  (99 frames of 20ms stride)
2. Context Representations (C) Shape : torch.Size([1, 99, 512])
3. Quantized Target Vectors (Q) Shape: torch.Size([1, 99, 512])
4. Masked Ti